# core

> Azure Data Explorer helper functions for notebook-based workflows

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import pandas as pd
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table


class ADXClient:
    """Thin wrapper around KustoClient with switchable authentication."""

    def __init__(self, cluster: str, interactive_login: bool = False):
        if not cluster:
            raise ValueError("Pass a cluster URL when creating an ADXClient.")
        self.cluster = cluster
        self.interactive_login = interactive_login
        self.credential = self._create_credential()
        self.client = self._create_client()

    def _create_credential(self):
        if self.interactive_login:
            return InteractiveBrowserCredential()
        return DefaultAzureCredential()

    def _create_client(self) -> KustoClient:
        kcsb = KustoConnectionStringBuilder.with_azure_token_credential(
            self.cluster, self.credential
        )
        return KustoClient(kcsb)

    def set_interactive_login(self, interactive_login: bool) -> None:
        self.interactive_login = interactive_login
        self.credential = self._create_credential()
        self.client = self._create_client()

    def perform_query(
        self,
        query: str | None = None,
        table: str = "Volve",
        database: str = "test",
    ) -> pd.DataFrame | None:
        """Run a KQL query and return the primary result table as a pandas DataFrame."""
        resolved_query = query or f"{table} | take 10"

        try:
            response = self.client.execute(database, resolved_query)
            return dataframe_from_result_table(response.primary_results[0])
        except Exception as exc:
            print(f"Query failed with error: {exc}")
            return None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()